# **Mejoras y hoja de ruta de los autoencoders**

## **0. Recapitulación de la serie y objetivo**

A lo largo de esta serie de notebooks hemos recorrido de forma controlada distintas decisiones de diseño sobre autoencoders para *morphing* facial en `64x64`:

- **`AE_complex1`:** línea base convolucional con latente de `128` dimensiones y pérdida `L1 + (1 - SSIM)` (`FID = 72.94`).
- **`AE_complex1_latent`:** ampliamos el latente a `256` dimensiones y confirmamos que el cuello de botella era una limitación real (`FID = 63.65`).
- **`AE_complex2`:** añadimos bloques residuales manteniendo el latente de `256`, pero el FID no mejoró (`64.40`) y vimos que el coste extra no compensaba.
- **`AE_complex2_VGG`:** incorporamos una pérdida perceptual temprana con `VGG16` congelada y conseguimos la mejor reconstrucción determinista en RGB (`FID = 61.17`).
- **`AE_complex3`:** escalamos el autoencoder a latente `512`, dos convoluciones por resolución y entrada en escala de grises (`21.4 M` de parámetros), comprobando que el coste computacional no compensaba la mejora marginal (`55.72` frente a `57.21` en gris).
- **`VAE_complex1` y `VAE_complex2`:** dimos el salto a autoencoders variacionales para regularizar el espacio latente, habilitar generación desde el prior (`FID = 87.43` con `z = 256`) y evaluar el morphing sobre `24` pares (`LERP` frente a `SLERP`, barrido de `z` y de `β`).

Una vez completados estos desarrollos, este documento va a servir como forma de centrar que hemos conseguido y hacia donde debemos apuntard, con el fin de atacar los dos grandes problemas pendientes: la **borrosidad de las reconstrucciones** (bajar de ese suelo de `FID ≈ 61-63`) y la **naturalidad de las transiciones**.

> **Alcance realista:** no podemos abarcarlo todo. Con nuestras restricciones de tiempo y cómputo, debemos priorizar qué líneas tienen mejor relación esfuerzo-beneficio. Al final, estamos realizando una tarea de investigación compleja y viene bien dejar algunas cosas claras: En primer lugar, nuestro objetivo está siendo obtener los mejores resultados posibles manteniendo una complejidad de cómputo estable y asumible. Crear mejores modelos sin tener esto en cuenta sería trivial, ya que bastaría con partir de uno de los modelos creados y escalarlo a la vez que la resolución de las imágenes. Sin embargo, nuestro objetivo es pensar y tomar pasos de forma justificada, no dar palos de ciego y mejorar el resultado por simple fuerza bruta.

A continuación vamos a explicar todas las ramas de mejora en **8 grandes bloques estratégicos**, indicando cuáles damos por cerrados o descartados y en cuáles vamos a centrar nuestros próximos pasos.

---

## **Mapa general: los 8 bloques estratégicos**

| Bloque | Enfoque | Estado en nuestra hoja de ruta |
| --- | --- | --- |
| **1. Evolución de la función de pérdida** | Cambiar qué penaliza la red | **Ya tratado** en la serie (cerrado por ahora) |
| **2. Mejoras en la arquitectura base** | Cambiar cómo comprime y decodifica | **Prioridad inmediata (siguiente desarrollo)** |
| **3. Regularización y topología latente** | Evitar agujeros en el espacio latente | **Descartado por tiempo** (avance sobre el VAE) |
| **4. Entrenamiento directo para interpolación** | Entrenar sobre mezclas latentes | **Descartado por complejidad** (similar a GAN) |
| **5. Arquitecturas híbridas (AE/VAE + GAN)** | Añadir un discriminador al decoder | **Próximo paso tras el Bloque 2** |
| **6. GANs puras e inversión latente** | Priorizar realismo y proyectar fotos | **Próximo paso junto con el Bloque 5** |
| **7. Modelos de difusión con latente** | Decoders de difusión condicional | **Estudiable:** depende de coste computacional |
| **8. Matemáticas de la trayectoria** | Cómo viajar de $z_1$ a $z_2$ | **Ya probado** (`LERP` vs `SLERP`, cerrado) |

---

## **Bloque 1. Evolución de la función de pérdida (*Loss Engineering*)**

*Estado: **ya tratado en la serie**. No empezamos directamente en `L1 + (1 - SSIM) + VGG16`, sino que ha sido un desarrollo progresivo que parte de analizar el comportamiento de **MSE ($L_2$)** y **MAE ($L_1$)** en `AE_complex1`, pasa por combinar `L1 + (1 - SSIM)` para proteger luminancia, contraste y estructura local, y termina con `AE_complex2_VGG` incorporando activaciones tempranas de `VGG16` para recuperar texturas.*

Si quisiéramos seguir profundizando únicamente en la función de pérdida sin alterar la arquitectura ni añadir redes competidoras, quedarían tres vías por explorar:

- **Pérdidas perceptuales específicas de dominio:** sustituir `VGG16` (entrenada en objetos generales de ImageNet a `224x224`) por redes entrenadas específicamente en rostros.
- **Pérdidas de identidad y atributos:** usar un extractor facial preentrenado para forzar que el rostro reconstruido (o interpolado) mantenga los rasgos del original.
- **Pérdidas en el dominio de la frecuencia:** penalizar directamente la pérdida de altas frecuencias (mechones de pelo, bordes nítidos) mediante transformadas de Fourier para prevenir la borrosidad típica de los autoencoders.

En general, cada una de estas tres líneas es muy interesante y directamente podría ser un único trabajo de investigación. Sin embargo, nuestro objetivo es mucho más amplio y no podemos permitirnos profundizar en todas ellas. Por ahora damos este bloque por suficientemente cubierto para centrarnos en mejoras estructurales.

---

## **Bloque 2. Mejoras en la arquitectura base (Encoder / Decoder)**

*Estado: **prioridad inmediata**. Es la vía de desarrollo que intentaremos abordar ahora mismo.*

En nuestra arquitectura actual comprimimos un mapa espacial de $4 \times 4 \times 256$ (`4096` valores) a un vector plano de `256` dimensiones mediante `Flatten()` y `Dense(256)` (lo que consume por sí solo más de un millón de parámetros en el encoder y otro millón en el decoder), reconstruimos con `UpSampling2D(bilinear)` y regularizamos con `l2(1e-5)`. Aunque los bloques residuales simples de `AE_complex2` no nos dieron resultado, hay cuatro mejoras estructurales que podrían ayudarnos:

### **2.1 Cuello de botella espacial (*Spatial Bottleneck*)**

- **El problema actual:** al usar `Flatten()` seguido de `Dense(256)`, destruimos la topología 2D de la imagen. Cada neurona del vector latente queda conectada a toda la imagen a la vez; al interpolar dos vectores, los rasgos pueden mezclarse de forma desordenada.
- **La solución:** sustituir `Flatten` + `Dense(256)` por una convolución $1 \times 1$ con `16` filtros sobre la última capa de $4 \times 4 \times 256$. Obtenemos un latente de forma `(4, 4, 16)`, que sigue teniendo exactamente **256 valores**, pero conserva la cuadrícula espacial.

#### **Convoluciones $1 \times 1$ frente a ResNet / Inception**

Es el mismo mecanismo matemático (proyección lineal a lo largo del eje de canales manteniendo las dimensiones espaciales $H \times W$), pero en nuestro autoencoder cumpliría una **doble función**:

1. **Reducción de parámetros (igual que en ResNet/Inception):** pasar de `(4, 4, 256)` a `Flatten()` (`4096` valores) y luego a `Dense(256)` genera una matriz de $4096 \times 256 = 1\,048\,576$ parámetros en el encoder y otros tantos en el decoder. Si aplicamos `Conv2D(16, kernel_size=1)` sobre `(4, 4, 256)`, los mismos $256 \times 16 = 4096$ pesos se reutilizan en cada una de las `16` posiciones de la cuadrícula $4 \times 4$. Pasamos de **~2.1 millones** de parámetros en el cuello de botella a apenas **~8 200**.
2. **Sesgo espacial (específico para morphing):** en ResNet o Inception se usa por eficiencia computacional; aquí, además, evita el mezclado global. Una capa `Dense` conecta el ojo izquierdo con la barbilla. Con la convolución $1 \times 1$, cada celda de la cuadrícula $4 \times 4$ comprime exclusivamente la información de su zona, haciendo que las interpolaciones sean geométricamente mucho más estables.

### **2.2 Bloques de atención espacial: *Self-Attention* frente a *CBAM***

- **Por qué importa en morphing:** las convoluciones $3 \times 3$ solo miran píxeles vecinos. Si interpolamos una cara con gafas hacia una sin gafas, una convolución pura puede dibujar media gafa en un ojo y nada en el otro.

- **Self-Attention (mecanismo Transformer):** es la misma operación *Query-Key-Value* (QKV) de los Transformers, adaptada a imágenes. Si la colocamos en la capa de resolución $8 \times 8$, tratamos esa imagen como una secuencia de $8 \times 8 = 64$ «tokens» (cada uno con sus `128` canales) y cada celda calcula su similitud con las otras `63`. Permite una conexión directa en un solo paso para coordinar la **simetría** entre rasgos distantes (ojo izquierdo y derecho, ambas orejas, gafas o dirección de la mirada). Como $64 \times 64$ es una matriz pequeña, su coste en esa resolución es asumible.
- **CBAM (*Convolutional Block Attention Module*):** **no** usa arquitectura Transformer ni QKV; es únicamente convolucional y muy barato en parámetros. Toma un mapa de características (por ejemplo, `(16, 16, 64)`) y aplica dos filtros secuenciales:
  1. **Atención de canal:** resume cada mapa con *Global Average Pooling* y *Max Pooling* para decidir *qué* filtros importan más en esa imagen.
  2. **Atención espacial:** colapsa los canales y pasa una sola convolución de $7 \times 7$ para generar un mapa de calor 2D de $16 \times 16$ con valores en $[0, 1]$ (mediante `sigmoid`), colapsando el fondo irrelevante y amplificando los rasgos faciales más importantes.

---

## **Bloque 3. Regularización y topología del espacio latente**

*Estado: **no lo abordaremos por tiempo**. Aunque es una vía muy interesante, supone un avance adicional sobre el mismo concepto de regularización del VAE que ya hemos trabajado en `VAE_complex1` y `VAE_complex2` (y en la normalización L2 mencionada en `AE_complex2_VGG`).*

En un autoencoder determinista sin restricciones fuertes (solo `l2(1e-5)`), el encoder coloca cada cara donde le resulta más cómodo para minimizar el error de píxel, dejando zonas muertas entre grupos de caras. Al interpolar en línea recta, cruzamos ese vacío y el decoder genera superposiciones fantasma. Las alternativas dentro de este bloque serían:

- **WAE-MMD (*Wasserstein Autoencoder* con *Maximum Mean Discrepancy*):** en un VAE clásico forzamos a que *cada* imagen individual tenga una distribución gaussiana alrededor de su media, lo que hace que las nubes de distintas caras se solapen y provoca borrosidad. El WAE-MMD mantiene el encoder **determinista** ($z = E(x)$), conservando la nitidez de un AE, pero añade una pérdida sobre todo el *batch* que mide la distancia (mediante un kernel *Inverse Multiquadratics*) entre la nube de puntos codificados y $\mathcal{N}(0, I)$. La distancia de Wasserstein es la métrica que se suele usar para hacer morphing de forma más matemática entre distribuciones.
- **$\beta$-VAE con KL dinámico (*Cyclical KL Annealing*):** entrena en ciclos donde $\beta = 0$ en la primera mitad (el modelo aprende  como un AE al inicio) y sube progresivamente hasta $\beta_{\max}$ en la segunda mitad (empaquetando el espacio latente), evitando que la KL apague dimensiones al principio.
- **Proyección sobre hiperesfera (*L2-Normalized Latent Space*):** fija la norma de los latentes ($z_{\text{norm}} = R \cdot z / \|z\|_2$) antes del decoder para que todas las caras se encuentren sobre una misma hiperesfera y la interpolación esférica (SLERP) sea exacta.

---

## **Bloque 4. Entrenamiento directo para interpolación**

*Estado: **descartado**. Tiene una dinámica adversaria muy similar a una GAN y añade una complejidad de ajuste y estabilidad difícil de asumir con nuestros recursos actuales.*

El fallo conceptual de entrenar un AE tradicional solo con $x \to E(x) \to D(z) \to \hat{x}$ es que **el decoder jamás ve un vector interpolado durante el entrenamiento**; solo aprende a decodificar puntos exactos del conjunto de entrenamiento. Para corregirlo existen tres ideas principales:

- **ACAI (*Adversarially Constrained Autoencoder Interpolation*):** en cada paso se toman dos imágenes ($x_1, x_2$), se obtienen sus latentes ($z_1, z_2$), se elige un peso aleatorio $\alpha \in [0, 0.5]$ y se decodifica el punto intermedio $\hat{x}_\alpha = D(\alpha z_1 + (1 - \alpha) z_2)$. Se añade una red **Crítico** ($C$) entrenada para predecir el factor $\alpha$ a partir de una imagen, mientras el autoencoder intenta engañar al Crítico haciendo que prediga $\alpha = 0$ para $\hat{x}_\alpha$. Si el Crítico no distingue una cara interpolada de una real, desaparece el efecto fantasma. En la práctica, este juego min-max comparte las dificultades de estabilidad e hiperparámetros de una GAN, pero sin el beneficio directo de un discriminador de realismo.
- **Consistencia cíclica en el latente:** volver a codificar la imagen intermedia $\hat{z}_\alpha = E(\hat{x}_\alpha)$ y añadir la pérdida $\|\hat{z}_\alpha - z_\alpha\|_1$ para que el decoder no destruya información semántica a mitad de camino.
- **Supervisión semántica con atributos (`attrs.npy`):** entrenar una clasificador auxiliar sobre el latente aprovechando los atributos de CelebA que ya cargamos, para forzar que el espacio sea linealmente separable respecto a sonrisa, gafas, género o color de pelo.

---

## **Bloques 5 y 6: modelos híbridos (AE/VAE + GAN) y GANs puras**

*Estado: **próximo gran paso tras probar el Bloque 2**. Una vez exploremos cuánto podemos mejorar la arquitectura base del autoencoder, la vía más directa para eliminar la borrosidad intrínseca de las pérdidas de píxel es introducir el aprendizaje adversario.*

Los autoencoders y VAEs tradicionales tienden a promediar los píxeles ante incertidumbre, generando resultados poco nítidos, mientras que las GANs destacan precisamente por producir texturas realistas y definidas. Para nuestro caso de uso (extraer latentes de dos caras reales, interpolar y decodificar), podemos abordarlo de dos formas:

### **Bloque 5. Arquitecturas híbridas (AE/VAE + GAN)**

Consisten en mantener nuestro flujo de codificación ($x \to z$) y añadir un **Discriminador** al final del decoder como una función de pérdida aprendida que penalice las reconstrucciones borrosas:

- **VAE-GAN (la mejora más directa sobre nuestro código actual):** mantiene la arquitectura de nuestro VAE (`Encoder` + `Decoder` de `VAE_complex1 o 2`), pero acopla un Discriminador de GAN a la salida del decoder. La pérdida combina la de nuestro VAE (reconstrucción + KL) con la pérdida adversaria, forzando detalles finos sin perder la suavidad del espacio latente.
- **AAE (*Adversarial Autoencoder*):** en este caso, el discriminador no evalúa las imágenes de salida, sino el **propio espacio latente**, compitiendo contra el encoder para que los vectores codificados encajen con una distribución previa (como una gaussiana) y evitar zonas vacías al interpolar.
- **Discriminadores locales (*PatchGAN* / estilo `pix2pix`) y *Feature Matching*:** utilizar el autoencoder como generador combinando la pérdida de reconstrucción con un discriminador que evalúe parches locales de $N \times N$ píxeles para forzar nitidez local, estabilizando el entrenamiento mediante *Feature Matching Loss* (comparar activaciones internas del discriminador entre la imagen real y la reconstruida).

### **Bloque 6. GANs puras e inversión del espacio latente (*GAN Inversion*)**

Consiste en invertir el paradigma: priorizar un generador que produzca caras con un FID muy bajo (como **DCGAN**, **ProGAN** o **StyleGAN**) y resolver a posteriori cómo proyectar nuestras dos imágenes de `test` hacia su espacio latente:

- **Inversión sobre generador preentrenado (*GAN Inversion*):** una GAN clásica no tiene encoder nativo para fotos reales. Se resuelve entrenando primero la GAN con el dataset de rostros y luego invirtiendo las dos imágenes objetivo: bien mediante **optimización por descenso de gradiente** (buscando el $z$ o $w \in W/W^+$ que haga que el generador reproduzca la foto), o bien entrenando un **encoder externo** (como *pSp* o *e4e*) con el generador congelado. Una vez obtenidos ambos latentes, interpolamos entre ellos.

---

## **Bloque 7. Modelos basados en difusión con espacio latente semántico**

*Estado: **a falta de determinar si es viable computacionalmente**, aunque representa la alternativa moderna del estado del arte para lograr un FID muy bajo sin sufrir inestabilidades como el *mode collapse*.*

- **Diffusion Autoencoders (DiffAE):** combinan un encoder semántico (que extrae un vector compacto de `256d` o `512d`) con un decoder basado en un modelo de difusión condicional que reconstruye la textura con mucha más nitidez.
- **Latent Diffusion Models (LDM):** utilizar nuestro AE/VAE actual para comprimir la imagen y aplicar un proceso de difusión directamente sobre los vectores latentes de `256` dimensiones.

---

## **Bloque 8. Matemáticas de la trayectoria de interpolación (post-procesado)**

*Estado: **ya probado en la serie (cerrado)**. No volveremos a tocar este bloque.*

Independientemente del modelo entrenado, la forma de viajar de $z_1$ a $z_2$ afecta al resultado:

- **Geometría del espacio (`LERP` frente a `SLERP`):** pasar de interpolación lineal (que atraviesa el interior perdiendo norma/varianza) a interpolación esférica. Ya implementamos y evaluamos esta comparación exhaustivamente en `AE_complex2_VGG` (mencionada en la sección 10.2), `VAE_complex1` y sobre los `24` pares de `VAE_complex2`: confirmamos que `SLERP` conserva la norma y da puntos medios algo más nítidos, pero produce pasos menos regulares y no mejora de forma medible el FID de los fotogramas, por lo que no resuelve el problema de fondo.
- **Caminos geodésicos (*Riemannian / Geodesic Interpolation*):** calcular la curva más corta siguiendo las regiones de mayor densidad de los datos aprendidos, rodeando zonas vacías.
- **Flujo óptico híbrido (*Spatial + Latent Morphing*):** combinar deformación geométrica en el espacio de píxeles con la transición latente para alinear ojos, nariz y boca durante el morphing.

---

## **Resumen de próximos pasos**

Hay algo que debe ser muy claro: a pesar de que haya bloques que no vamos a tocar porque hayamos determinado que nos han dado peores resultados, podría darse el caso de que simplemente el modelo o técnica usada no estén optimizadas para el trabajo que queremos hacer. Por ejemplo, que los bloques residuales de `AE_complex2` no nos hayan dado resultado no significa que los **bloques residuales** sean malos para `morphing`, sino que la forma en la que los implementamos no era la adecuada para nuestro caso de uso. Esto puede ser porque no tenmos datos suficientes, resolución, o porque la arquitectura no estaba bien ajustada. Un ejemplo más claro de esto podría ser un **transformer**, que usado con pocos datos puede dar resultados peores que una **CNN** para multitud de tareas, pero nadie toma en duda que, en su situación adecuada, pueden llegar a ser mucho mejores.

1. **Corto plazo (Bloque 2):** experimentar con mejoras estructurales en el propio autoencoder sin salir del paradigma AE/VAE.
2. **Medio plazo (Bloques 5 y 6):** dar el salto al entrenamiento adversario mediante un modelo híbrido (**VAE-GAN** o **PatchGAN** sobre nuestro encoder/decoder) y compararlo con la **inversión de GANs puras**, atacando la borrosidad que limita a las pérdidas de reconstrucción actuales.